# 05 — Self-attention deep dive

We build causal self-attention step by step on a toy batch, then show why scores are
scaled by `1/√d_k`. See `docs/04_transformer_and_gpt.md` for the theory.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

## Step 1 — averaging the past with a triangular matrix
The simplest way for token *t* to use its context: average tokens `0..t`. A lower
triangular weight matrix does this for all positions in one matmul.

In [2]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)
xbow = wei @ x                           # (T,T) @ (B,T,C) -> (B,T,C)

# check against a loop
xbow_loop = torch.zeros_like(x)
for b in range(B):
    for t in range(T):
        xbow_loop[b, t] = x[b, :t + 1].mean(0)
print(torch.allclose(xbow, xbow_loop, atol=1e-6))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
True


## Step 2 — data-dependent weights: queries, keys, values
Instead of uniform weights, each token's query is compared with every key.

In [3]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                               # (B, T, hs)
q = query(x)                             # (B, T, hs)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, T)
wei = wei.masked_fill(tril == 0, float('-inf'))     # decoder: no peeking at the future
wei = F.softmax(wei, dim=-1)
v = value(x)
out = wei @ v                            # (B, T, hs)
print(out.shape)
print(wei[0].round(decimals=2))

torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3100, 0.2900, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.2200, 0.2400, 0.2100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2000, 0.1700, 0.1500, 0.3400, 0.0000, 0.0000, 0.0000],
        [0.1300, 0.2500, 0.1300, 0.1100, 0.3100, 0.0700, 0.0000, 0.0000],
        [0.1600, 0.2000, 0.1100, 0.1100, 0.1400, 0.1700, 0.1100, 0.0000],
        [0.0800, 0.1200, 0.1100, 0.1500, 0.1100, 0.1100, 0.1600, 0.1600]],
       grad_fn=<RoundBackward1>)


Notes:
* Attention is a **communication mechanism** — nodes in a directed graph aggregating
  information from the nodes that point to them.
* It has no notion of space, hence positional embeddings.
* Batch elements never talk to each other.
* Removing the mask gives an *encoder* block (every token sees all tokens).

## Why we scale by 1/√d_k

In [4]:
for d_k in [4, 16, 64, 256]:
    q = torch.randn(1000, d_k)
    k = torch.randn(1000, d_k)
    raw = (q * k).sum(-1)
    scaled = raw * d_k ** -0.5
    print(f'd_k={d_k:4d}  var(q·k)={raw.var():7.2f}   var(q·k/√d_k)={scaled.var():.2f}')

scores = torch.randn(8)
print('softmax(scores)      ', F.softmax(scores, dim=-1).round(decimals=3))
print('softmax(scores * 16) ', F.softmax(scores * 16, dim=-1).round(decimals=3), '<- almost one-hot')

d_k=   4  var(q·k)=   4.26   var(q·k/√d_k)=1.06
d_k=  16  var(q·k)=  15.94   var(q·k/√d_k)=1.00
d_k=  64  var(q·k)=  63.65   var(q·k/√d_k)=0.99
d_k= 256  var(q·k)= 252.52   var(q·k/√d_k)=0.99
softmax(scores)       tensor([0.2510, 0.0370, 0.0780, 0.0570, 0.0750, 0.1030, 0.3870, 0.0110])
softmax(scores * 16)  tensor([0.0010, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9990, 0.0000]) <- almost one-hot
